# 00 — Environment and DB Check

Confirm the environment, config, DB access, and writable output root before the experiment.


In [ ]:
from pathlib import Path
import os
import re
import sys
import tempfile
import logging
from textwrap import indent

import yaml

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'code_base').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / 'code_base' / 'fea_cad_one_sample'
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / 'src' / 'api.env'


def _load_api_env(path: Path) -> dict[str, str]:
    """Load a local api.env file into the current process environment."""

    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values


api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api
from src import runners

logging.basicConfig(level=logging.INFO, format='%(name)s | %(levelname)s | %(message)s')

FIXED_SAMPLE_PATH = MODULE_ROOT / 'experiment_config' / 'fixed_sample.yaml'
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding='utf-8')) if FIXED_SAMPLE_PATH.exists() else {}
sample_id = str(sample_config.get('sample_id', '00689964'))
selection_source = str(sample_config.get('load_source', 'db'))
dataset_original_dir = MODULE_ROOT / 'outputs' / f'sample_{sample_id}' / '01_dataset_original'

print('[STAGE] Paths')
print(f'  → python      : {sys.executable}')
print(f'  → project root: {PROJECT_ROOT}')
print(f'  → module root : {MODULE_ROOT}')
print(f'  → outputs     : {MODULE_ROOT / "outputs"}')
print(f'  → config dir  : {MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs"}')
print(f'  → api.env     : {API_ENV_PATH}')
print(f'  → selection   : {selection_source}')
print(f'  → dataset dir : {dataset_original_dir}')


In [ ]:
print('[STAGE] Dependency imports')
import cadquery as cq
import matplotlib
import matplotlib.pyplot as plt
import PIL.Image
import trimesh

print('  ✓ cadquery imported')
print('  ✓ matplotlib imported')
print('  ✓ pillow imported')
print('  ✓ trimesh imported')
print(f'  → runners module : {runners.__name__}')
print(f'  → interfaces API : {api.__name__}')


In [ ]:
print('[STAGE] Config and env check')
config_path = MODULE_ROOT / 'src' / 'copied_from_cadcodeverify' / 'configs' / 'config_gpt_5_4_mini.yaml'
config_text = config_path.read_text(encoding='utf-8')
config_data = yaml.safe_load(config_text)
required_env_vars = sorted(set(re.findall(r'\$\{([A-Z_][A-Z0-9_]*)\}', config_text)))
if selection_source == 'db' and 'CAD_DB_CONNECTION_STRING' not in required_env_vars:
    required_env_vars.append('CAD_DB_CONNECTION_STRING')
env_status = {name: bool(os.environ.get(name)) for name in required_env_vars}

print(f'  → config file : {config_path}')
print(f'  → config keys : {sorted(config_data.keys())}')
print(f'  → env vars    : {env_status}')
print(f'  → api.env keys : {sorted(api_env_values)}')
print('  ✓ config loaded without printing secrets')


In [ ]:
print('[STAGE] DB schema inspection')
if selection_source == 'dataset':
    metadata_path = dataset_original_dir / 'metadata.json'
    original_prompt_path = dataset_original_dir / 'original_prompt.txt'
    original_code_path = dataset_original_dir / 'database_original_code.py'
    assert metadata_path.exists()
    assert original_prompt_path.exists()
    assert original_code_path.exists()
    print(f'  → metadata : {metadata_path}')
    print(f'  → prompt   : {original_prompt_path}')
    print(f'  → code     : {original_code_path}')
    print('  ✓ dataset artifacts are available')
else:
    connection_string = os.environ['CAD_DB_CONNECTION_STRING']
    schema = api.inspect_schema(connection_string)
    table_names = [table['name'] for table in schema['tables']]
    print(f'  → dialect : {schema["dialect"]}')
    print(f'  → tables  : {table_names}')

    interesting_tables = {'master_metadata', 'ground_truth_code'}
    for table_name in sorted(interesting_tables):
        table = next(item for item in schema['tables'] if item['name'] == table_name)
        column_names = [column['name'] for column in table['columns']]
        print(f'  → {table_name}: {column_names}')

    master = next(item for item in schema['tables'] if item['name'] == 'master_metadata')
    truth = next(item for item in schema['tables'] if item['name'] == 'ground_truth_code')
    master_columns = {column['name'] for column in master['columns']}
    truth_columns = {column['name'] for column in truth['columns']}
    assert {'id', 'expert_prompt'}.issubset(master_columns)
    assert {'id', 'python_code'}.issubset(truth_columns)
    print('  ✓ expert prompt and original CAD code are discoverable')


In [ ]:
print('[STAGE] Writable output root')
output_root = MODULE_ROOT / 'outputs'
output_root.mkdir(parents=True, exist_ok=True)
probe = output_root / '.write_probe'
probe.write_text('ok', encoding='utf-8')
assert probe.exists()
probe.unlink()
print(f'  → output root : {output_root}')
print('  ✓ output path writable')


## What this notebook proves

- Environment is ready.
- DB is reachable.
- Config is loaded.
- Output root is writable.
